# Deep HANK Exact-CHI Runner (VS Code + Colab Runtime)

Primary workflow:
1. Open this notebook in VS Code.
2. Select a **Colab hosted runtime** from the kernel selector.
3. Run cells top-to-bottom.

This notebook keeps code in GitHub, runs training on Colab GPU, and writes outputs/checkpoints to Google Drive.

In [ ]:
# ---- Run parameters ----
REPO_URL = "https://github.com/VicWu07/deep-hank-code-model-3-colab.git"
BRANCH = "main"
PROJECT_SUBDIR = "."

# One notebook session = one chi run. Launch multiple sessions for parallel chi jobs.
CHI = 1.0
TRAIN_EPOCHS = 50          # smoke test default; raise to 5000 for full run
CHECKPOINT_EVERY = 25      # e.g., 250/500 for full runs
LABEL = "smoke"

# Drive destination root (each run writes to a unique child folder)
OUTPUT_ROOT = "/content/drive/MyDrive/deep_hank_runs"

# ---- Setup controls (for repeated chi runs) ----
# Fast repeat-run defaults: reuse repo and skip pip install in the same runtime.
SYNC_CODE = False          # True: git fetch/pull latest; False: reuse current checkout
INSTALL_DEPS = False       # True on first run of a fresh runtime; False for repeat chi runs

# Where repo code lives during this runtime:
# - "ephemeral": /content/workspace/repo (faster, but disappears after runtime reset)
# - "drive": DRIVE_REPO_DIR (persistent and visible in Google Drive)
REPO_STORAGE = "ephemeral"
DRIVE_REPO_DIR = "/content/drive/MyDrive/deep_hank_repo"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pathlib
import subprocess

if REPO_STORAGE not in {"ephemeral", "drive"}:
    raise ValueError('REPO_STORAGE must be "ephemeral" or "drive"')

if REPO_STORAGE == "drive":
    repo_root = pathlib.Path(DRIVE_REPO_DIR)
else:
    repo_root = pathlib.Path('/content/workspace/repo')

repo_root.parent.mkdir(parents=True, exist_ok=True)

if not repo_root.exists():
    subprocess.run([
        'git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(repo_root)
    ], check=True)
    print('Cloned fresh repo checkout')
else:
    if SYNC_CODE:
        subprocess.run(['git', '-C', str(repo_root), 'fetch', 'origin', BRANCH], check=True)
        subprocess.run(['git', '-C', str(repo_root), 'checkout', BRANCH], check=True)
        subprocess.run(['git', '-C', str(repo_root), 'pull', '--ff-only', 'origin', BRANCH], check=True)
        print('Synced repo with remote branch')
    else:
        print('Reusing existing repo checkout (SYNC_CODE=False)')

project_dir = repo_root / PROJECT_SUBDIR
if not project_dir.exists():
    raise FileNotFoundError(f'Project subdir not found: {project_dir}')

print('Repo root   :', repo_root)
print('Project dir :', project_dir)
if REPO_STORAGE == 'ephemeral':
    print('Note: code is in /content (runtime disk), not Google Drive.')
    print('Set REPO_STORAGE="drive" if you want the repo visible/persistent in Drive.')

In [ ]:
import subprocess
req_file = project_dir / 'requirements-colab.txt'
if INSTALL_DEPS:
    subprocess.run(['python3', '-m', 'pip', 'install', '-U', 'pip'], check=True)
    subprocess.run(['python3', '-m', 'pip', 'install', '-r', str(req_file)], check=True)
    print('Installed requirements from', req_file)
else:
    print('Skipped pip install (INSTALL_DEPS=False).')
    print('Set INSTALL_DEPS=True for a fresh runtime or after dependency changes.')

In [ ]:
import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))
else:
    print('WARNING: GPU not detected. Change runtime to GPU before full runs.')

In [ ]:
import subprocess

launcher = project_dir / 'scripts' / 'run_exact_chi_colab.sh'
cmd = [
    'bash', str(launcher),
    '--chi', str(CHI),
    '--train-epochs', str(TRAIN_EPOCHS),
    '--checkpoint-every', str(CHECKPOINT_EVERY),
    '--output-root', OUTPUT_ROOT,
    '--label', LABEL,
    '--python', 'python3',
]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True, cwd=str(project_dir))

In [ ]:
# Quick tail of progress + output listing for this chi/label
import pathlib

chi_tag = f"{CHI:.3f}".replace('-', 'm').replace('.', 'p')
run_name = f"exact_chi_{chi_tag}" + (f"_{LABEL}" if LABEL else "")
run_dir = pathlib.Path(OUTPUT_ROOT) / f"{run_name}_outputs"
print('Run dir:', run_dir)

progress = run_dir / 'progress.txt'
if progress.exists():
    print('\n--- progress.txt ---')
    print(progress.read_text(encoding='utf-8'))
else:
    print('No progress.txt found yet')

print('\n--- top-level artifacts ---')
for p in sorted(run_dir.glob('*')):
    print(p.name)

In [ ]:
# Compare finished runs across chi values (summary JSON only)
import glob
import json
import os

rows = []
for summary_path in glob.glob(os.path.join(OUTPUT_ROOT, 'exact_chi_*_outputs', 'mrs_optimized_exact_chi_summary.json')):
    with open(summary_path, 'r', encoding='utf-8') as f:
        s = json.load(f)
    rows.append({
        'path': summary_path,
        'chi': s.get('config', {}).get('model', {}).get('chi'),
        'epochs_completed': s.get('training', {}).get('epochs_completed'),
        'terminal_loss': s.get('training', {}).get('terminal_loss'),
        'steady_mae': s.get('benchmark_errors', {}).get('steady_anchor', {}).get('mae'),
        'zlow_mae': s.get('benchmark_errors', {}).get('z_low', {}).get('mae'),
        'zhigh_mae': s.get('benchmark_errors', {}).get('z_high', {}).get('mae'),
    })

rows = sorted(rows, key=lambda x: (float(x['chi']) if x['chi'] is not None else 1e9))
for r in rows:
    print(r)